# Matplotlib


This notebook is a practical learning and reference guide for Matplotlib, aimed at analysts, ML engineers, and data-oriented software engineers. It focuses on the plotting habits that matter in real work: choosing the right chart, controlling layout, labeling clearly, avoiding misleading visuals, and exporting figures that are ready for reports or presentations.


## 1. Introduction to Matplotlib

Matplotlib is Python's foundational plotting library. It sits lower in the visualization stack than Seaborn or pandas plotting, which is exactly why it still matters: it gives you direct control over figures, axes, labels, legends, annotations, scales, and export settings.

Matplotlib is especially useful when you need:
- precise control over plot structure and styling
- custom layouts with multiple panels
- publication-ready or presentation-ready figures
- plotting logic that should stay stable across notebooks, scripts, and applications

A useful mental model is:
- NumPy provides arrays and numerical computation
- pandas provides labeled tabular data
- Seaborn provides statistical plotting shortcuts
- Matplotlib provides the underlying drawing system and fine-grained control

Quick plots are great for exploration. Final plots for communication need more judgment: clear labeling, deliberate scales, sensible color choices, and enough polish that the figure explains the data instead of distracting from it.


## 2. Setup and Plotting Environment

Reusing the same small datasets across sections builds intuition better than inventing a brand-new example each time. This notebook sets up a few compact tables and arrays that we will return to repeatedly.

Two practical habits are worth establishing early:
- set seeds when randomness is involved so examples are reproducible
- understand that plots are built from a **Figure** containing one or more **Axes**, and each Axes holds artists such as lines, bars, text, legends, ticks, and annotations


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, PercentFormatter
import matplotlib.dates as mdates

np.random.seed(42)
rng = np.random.default_rng(42)

plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

x = np.linspace(0, 10, 200)
y = np.sin(x)
y_cos = np.cos(x)

sales_dates = pd.date_range("2025-01-01", periods=24, freq="W")
sales_df = pd.DataFrame(
    {
        "date": sales_dates,
        "sales": 180 + np.linspace(0, 70, len(sales_dates)) + rng.normal(0, 12, len(sales_dates)),
        "target": 235,
    }
)
sales_df["rolling_4"] = sales_df["sales"].rolling(4).mean()

category_df = pd.DataFrame(
    {
        "team": ["North", "South", "East", "West"],
        "revenue": [420, 360, 390, 450],
        "profit_margin": [0.24, 0.18, 0.22, 0.27],
        "tickets": [120, 155, 133, 148],
    }
)

survey_df = pd.DataFrame(
    {
        "segment": ["SMB", "Mid-Market", "Enterprise"],
        "q1": [42, 35, 23],
        "q2": [45, 33, 22],
        "q3": [40, 37, 23],
    }
)

distribution_df = pd.DataFrame(
    {
        "control": rng.normal(50, 8, 300),
        "variant_a": rng.normal(55, 10, 300),
        "variant_b": rng.normal(52, 6, 300),
    }
)

scatter_df = pd.DataFrame(
    {
        "study_hours": rng.uniform(1, 12, 60),
        "exam_score": 48 + rng.uniform(1, 12, 60) * 4 + rng.normal(0, 8, 60),
        "group": rng.choice(["A", "B", "C"], 60, p=[0.4, 0.35, 0.25]),
    }
)

heatmap_data = rng.normal(size=(8, 8))

export_dir = Path("matplotlib_playground/exports")
export_dir.mkdir(parents=True, exist_ok=True)

print("Figure contains Axes; Axes contain artists like lines, bars, ticks, text, legends, and annotations.")
print("Reusable datasets:", ["sales_df", "category_df", "survey_df", "distribution_df", "scatter_df", "heatmap_data"])


## 3. Your First Plot

Start small. A first plot should build confidence, not try to demonstrate every option at once.

The default line plot is perfectly fine for exploration. For communication, though, you will usually want at least:
- a title
- axis labels
- a legend when multiple series appear
- enough figure size and resolution that labels are easy to read


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(x, y, label="sin(x)")
plt.title("A first Matplotlib line plot")
plt.xlabel("x")
plt.ylabel("sin(x)")
plt.legend()
plt.show()


## 4. The Figure-Axes Model

This is one of the most important concepts in Matplotlib.

- A **Figure** is the whole canvas.
- An **Axes** is one plotting area within that canvas.
- The stateful `plt.plot(...)` style is convenient for quick work.
- The object-oriented style `fig, ax = plt.subplots()` is usually better for larger notebooks, reusable code, and multi-panel figures.

In practice, prefer `ax.plot(...)`, `ax.set_title(...)`, and related Axes methods once plots become even slightly complex.


In [ ]:
# Stateful interface: quick for one-off exploration
plt.figure(figsize=(6, 3.5))
plt.plot(x, y, color="steelblue")
plt.title("Stateful interface")
plt.show()

# Object-oriented interface: preferred for reusable plotting logic
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(x, y, color="tomato", label="sin(x)")
ax.set_title("Object-oriented interface")
ax.set_xlabel("x")
ax.set_ylabel("value")
ax.legend()
plt.show()

print("Figure object:", type(fig).__name__)
print("Axes object:", type(ax).__name__)


## 5. Basic Plot Types

A good plotting habit is to think in pairs: **syntax plus judgment**.

For each chart type, ask:
- what data shape is this good for?
- what comparison does it support well?
- what is the most common misuse?

Quick guidance:
- line plots: trends or ordered sequences
- scatter plots: relationships between two continuous variables
- bar charts: comparing discrete categories
- histograms: distributions of continuous variables
- box plots: compact distribution summaries
- area plots: cumulative or total emphasis over an ordered axis
- pie charts: only for very small part-to-whole comparisons, and even then often replaceable with bars


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7), constrained_layout=True)

# line
axes[0, 0].plot(x, y, color="royalblue", linewidth=2)
axes[0, 0].set_title("Line: ordered trend")

# scatter
group_colors = {"A": "tab:blue", "B": "tab:orange", "C": "tab:green"}
for group, subset in scatter_df.groupby("group"):
    axes[0, 1].scatter(
        subset["study_hours"],
        subset["exam_score"],
        alpha=0.75,
        s=45,
        label=group,
        color=group_colors[group],
    )
axes[0, 1].set_title("Scatter: relationship")
axes[0, 1].legend(title="group", frameon=False)

# bar
axes[0, 2].bar(category_df["team"], category_df["revenue"], color="slateblue")
axes[0, 2].set_title("Bar: category comparison")

# horizontal bar
sorted_margin = category_df.sort_values("profit_margin")
axes[0, 3].barh(sorted_margin["team"], sorted_margin["profit_margin"], color="seagreen")
axes[0, 3].xaxis.set_major_formatter(PercentFormatter(1.0))
axes[0, 3].set_title("Barh: long labels")

# histogram
axes[1, 0].hist(distribution_df["variant_a"], bins=16, color="goldenrod", edgecolor="white")
axes[1, 0].set_title("Histogram: distribution")

# box plot
axes[1, 1].boxplot(
    [distribution_df["control"], distribution_df["variant_a"], distribution_df["variant_b"]],
    labels=["control", "A", "B"],
)
axes[1, 1].set_title("Box plot: summary")

# area / fill
axes[1, 2].fill_between(sales_df["date"], sales_df["sales"], color="skyblue", alpha=0.6)
axes[1, 2].set_title("Area: volume emphasis")
axes[1, 2].tick_params(axis="x", rotation=35)

# pie
axes[1, 3].pie(category_df["revenue"], labels=category_df["team"], autopct="%1.0f%%", startangle=90)
axes[1, 3].set_title("Pie: use sparingly")

plt.show()


Common misuse reminders:
- use lines only when the x-axis has meaningful order
- use bars for categories, not for dense continuous data
- avoid pies when categories are numerous or values are close
- do not assume a box plot tells the full distribution story
- histogram conclusions can change a lot when the bin count changes


## 6. Customizing Lines, Markers, and Colors

Styling is where many beginner plots become less readable instead of more readable. A simple rule helps: add visual encodings only when they carry meaning.

Good habits:
- keep line widths and marker sizes readable but not heavy
- use alpha when overplotting would otherwise hide density
- avoid relying on color alone when categories must stay distinguishable
- do not combine too many differences in one line unless each one means something


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(x, y, color="navy", linestyle="-", linewidth=2.5, label="baseline")
ax.plot(x, y_cos, color="darkorange", linestyle="--", linewidth=2, marker="o", markevery=20, markersize=5, alpha=0.85, label="comparison")
ax.set_title("Styling lines without overwhelming the plot")
ax.set_xlabel("x")
ax.set_ylabel("value")
ax.legend(frameon=False)
ax.grid(alpha=0.25)
plt.show()


## 7. Titles, Labels, Ticks, and Axis Limits

Labeling is part of the analysis, not decoration. A plot with unclear units or unreadable ticks can be technically correct and still fail as communication.

Best practices:
- use titles to explain what the reader is looking at
- include units in axis labels when relevant
- rotate labels only as much as needed
- change limits carefully; aggressive axis trimming can mislead


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(category_df["team"], category_df["revenue"], color="cornflowerblue")
ax.set_title("Quarterly revenue by region")
ax.set_xlabel("Region")
ax.set_ylabel("Revenue (thousand USD)")
ax.set_ylim(0, 500)
ax.set_yticks(np.arange(0, 501, 100))
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"${int(v)}k"))
ax.tick_params(axis="x", rotation=0)
plt.show()


## 8. Legends and Gridlines

Legends and gridlines should support decoding the figure, not dominate it.

Practical habits:
- label series directly and clearly
- place legends where they do not hide important data
- keep gridlines subtle
- remove chart elements that add noise instead of structure


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(sales_df["date"], sales_df["sales"], label="weekly sales", color="tab:blue", linewidth=2)
ax.plot(sales_df["date"], sales_df["rolling_4"], label="4-week rolling mean", color="tab:red", linewidth=2.5)
ax.legend(loc="upper left", frameon=False)
ax.grid(axis="y", alpha=0.25)
ax.set_title("Legends and gridlines used for comparison")
ax.set_ylabel("Sales")
ax.tick_params(axis="x", rotation=35)
plt.show()


## 9. Subplots and Multi-Panel Figures

Multi-panel figures are central to real analysis. They let you compare categories, show different views of the same data, or place exploratory and summary views side by side.

Useful habits:
- choose a layout that matches the story
- use shared axes when comparisons should be direct
- flatten the axes array when looping is simpler than indexing with row and column pairs


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=False, sharey=False, constrained_layout=True)
axes = axes.ravel()

axes[0].plot(sales_df["date"], sales_df["sales"], color="tab:blue")
axes[0].set_title("Trend over time")
axes[0].tick_params(axis="x", rotation=35)

axes[1].hist(distribution_df["control"], bins=18, color="tab:gray", edgecolor="white")
axes[1].set_title("Distribution")

axes[2].bar(category_df["team"], category_df["tickets"], color="tab:green")
axes[2].set_title("Category comparison")

axes[3].scatter(scatter_df["study_hours"], scatter_df["exam_score"], alpha=0.7, color="tab:orange")
axes[3].set_title("Relationship view")

for ax in axes:
    ax.grid(alpha=0.15)

fig.suptitle("One dataset, multiple analytical views", fontsize=14)
plt.show()


## 10. Plotting Categorical Data

Categorical plotting is less about raw syntax and more about ordering, spacing, and label handling.

Good habits:
- sort categories intentionally rather than accepting arbitrary order
- annotate bars when exact values matter
- keep label text compact
- avoid crowded grouped bars when categories multiply too far


In [ ]:
ordered = category_df.sort_values("revenue", ascending=False)
xpos = np.arange(len(survey_df["segment"]))
width = 0.25

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)

bars = axes[0].bar(ordered["team"], ordered["revenue"], color="teal")
axes[0].set_title("Ordered bars improve readability")
axes[0].set_ylabel("Revenue (thousand USD)")
for bar in bars:
    axes[0].annotate(
        f"{bar.get_height():.0f}",
        (bar.get_x() + bar.get_width() / 2, bar.get_height()),
        ha="center",
        va="bottom",
        textcoords="offset points",
        xytext=(0, 4),
    )

axes[1].bar(xpos - width, survey_df["q1"], width=width, label="Q1")
axes[1].bar(xpos, survey_df["q2"], width=width, label="Q2")
axes[1].bar(xpos + width, survey_df["q3"], width=width, label="Q3")
axes[1].set_xticks(xpos)
axes[1].set_xticklabels(survey_df["segment"])
axes[1].set_title("Grouped bars for category-by-period comparisons")
axes[1].set_ylabel("Share of deals (%)")
axes[1].legend(frameon=False)

plt.show()


## 11. Plotting Distributions

Distributions deserve extra care because the chart design changes the interpretation quickly.

A few judgment rules:
- histogram binning changes the apparent shape
- box plots are compact summaries, not full distribution descriptions
- violin plots can reveal shape better, but they need enough data to be meaningful


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), constrained_layout=True)

axes[0].hist(distribution_df["control"], bins=10, alpha=0.75, label="10 bins", color="tab:blue")
axes[0].hist(distribution_df["control"], bins=25, alpha=0.45, label="25 bins", color="tab:orange")
axes[0].set_title("Histogram bin choice matters")
axes[0].legend(frameon=False)

axes[1].boxplot(
    [distribution_df["control"], distribution_df["variant_a"], distribution_df["variant_b"]],
    labels=["control", "A", "B"],
)
axes[1].set_title("Box plots summarize spread")

axes[2].violinplot(
    [distribution_df["control"], distribution_df["variant_a"], distribution_df["variant_b"]],
    showmeans=True,
    showextrema=False,
)
axes[2].set_xticks([1, 2, 3])
axes[2].set_xticklabels(["control", "A", "B"])
axes[2].set_title("Violin plots show shape")

plt.show()


## 12. Plotting Time Series

Time series plots become much clearer when date formatting is intentional. Dense date ticks, overlapping labels, and unclear smoothing choices are some of the most common readability problems.

Practical habits:
- keep the time axis ordered and parsed as datetime
- rotate labels only when needed
- use moving averages as context, not as a replacement for raw data
- highlight ranges or events when they matter to interpretation


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(sales_df["date"], sales_df["sales"], marker="o", markersize=4, linewidth=1.8, label="weekly sales")
ax.plot(sales_df["date"], sales_df["rolling_4"], linewidth=2.6, color="tab:red", label="4-week rolling mean")
ax.axhline(sales_df["target"].iloc[0], color="gray", linestyle="--", linewidth=1.5, label="target")
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.set_title("Weekly sales trend with target and rolling average")
ax.set_ylabel("Sales")
ax.legend(frameon=False, ncol=3, loc="upper left")
ax.tick_params(axis="x", rotation=35)
ax.grid(axis="y", alpha=0.2)
plt.show()


## 13. Annotations and Reference Lines

Plots become explanatory when you highlight meaningful events instead of forcing the reader to discover every insight alone.

Good annotation habits:
- annotate only the points that matter
- keep text concise
- use arrows, spans, and reference lines to clarify thresholds or events
- avoid covering the data with labels


In [ ]:
peak_idx = sales_df["sales"].idxmax()
peak_row = sales_df.loc[peak_idx]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(sales_df["date"], sales_df["sales"], color="tab:blue", linewidth=2)
ax.axhline(235, color="tab:red", linestyle="--", linewidth=1.5, label="target")
ax.axvspan(sales_df["date"].iloc[8], sales_df["date"].iloc[11], color="gold", alpha=0.2, label="campaign window")
ax.annotate(
    f"Peak: {peak_row['sales']:.0f}",
    xy=(peak_row["date"], peak_row["sales"]),
    xytext=(20, 20),
    textcoords="offset points",
    arrowprops={"arrowstyle": "->", "color": "black"},
)
ax.set_title("Reference lines and annotation focus attention")
ax.legend(frameon=False)
ax.tick_params(axis="x", rotation=35)
plt.show()


## 14. Layout, Text, and Visual Polish

A polished figure is usually the result of many small decisions rather than one dramatic styling trick.

Common polish tools:
- `tight_layout()` for quick spacing fixes
- `constrained_layout=True` for more automatic layout management
- `suptitle()` for figure-level titles
- spine visibility changes to reduce chart junk
- font-size adjustments for titles, labels, and annotations


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)

axes[0].plot(x, y, color="#1f77b4", linewidth=2.2)
axes[0].set_title("Default polish choices", fontsize=13)
axes[0].set_xlabel("x")
axes[0].set_ylabel("sin(x)")
axes[0].grid(alpha=0.2)

axes[1].scatter(scatter_df["study_hours"], scatter_df["exam_score"], color="#d62728", alpha=0.7)
axes[1].set_title("Readable spacing and restrained styling", fontsize=13)
axes[1].set_xlabel("Study hours")
axes[1].set_ylabel("Exam score")
axes[1].grid(alpha=0.2)

fig.suptitle("Polish is mostly about readability", fontsize=15)
plt.show()


## 15. Common Statistical and Analytical Plot Patterns

Many business and analytical plots repeat a few high-value patterns:
- trend line plus raw series
- rolling mean overlays
- baseline or target comparisons
- shaded bands for uncertainty or acceptable ranges
- cumulative totals for progress tracking

These are powerful because they help readers compare the data to a reference, not just stare at raw values.


In [ ]:
trend_coef = np.polyfit(scatter_df["study_hours"], scatter_df["exam_score"], 1)
trend_line = np.poly1d(trend_coef)

sales_df["cumulative_sales"] = sales_df["sales"].cumsum()
band_center = sales_df["rolling_4"].bfill()
lower_band = band_center - 12
upper_band = band_center + 12

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)

axes[0].scatter(scatter_df["study_hours"], scatter_df["exam_score"], alpha=0.65, color="tab:blue")
axes[0].plot(
    np.sort(scatter_df["study_hours"]),
    trend_line(np.sort(scatter_df["study_hours"])),
    color="tab:red",
    linewidth=2.5,
)
axes[0].set_title("Scatter with fitted trend line")
axes[0].set_xlabel("Study hours")
axes[0].set_ylabel("Exam score")

axes[1].plot(sales_df["date"], sales_df["cumulative_sales"], color="tab:green", linewidth=2.5, label="cumulative sales")
axes[1].fill_between(sales_df["date"], lower_band, upper_band, color="tab:blue", alpha=0.15, label="rolling range")
axes[1].plot(sales_df["date"], band_center, color="tab:blue", linewidth=2, label="rolling mean")
axes[1].set_title("Cumulative trend with contextual band")
axes[1].legend(frameon=False)
axes[1].tick_params(axis="x", rotation=35)

plt.show()


## 16. Images, Heatmaps, and Matrix Visuals

Matplotlib is not limited to lines and bars. It is also useful for image-like and matrix-like data.

When using `imshow`:
- choose colormaps deliberately
- keep aspect ratio in mind
- use a colorbar when the magnitude scale matters
- be careful with interpolation because it can smooth or imply structure that is not really there


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
image = ax.imshow(heatmap_data, cmap="coolwarm", aspect="auto", interpolation="nearest")
ax.set_title("Heatmap-style matrix with imshow")
ax.set_xlabel("Feature index")
ax.set_ylabel("Observation index")
fig.colorbar(image, ax=ax, shrink=0.85, label="z-score")
plt.show()


## 17. Log Scales and Axis Transformations

Log scales are useful when values span orders of magnitude, but they can also confuse readers when used casually.

Use log scales when:
- multiplicative differences matter more than additive ones
- growth behaves roughly exponentially
- values vary over very wide ranges

Pitfalls:
- zero and negative values do not work on standard log axes
- readers may misread equal visual spacing as equal absolute differences


In [ ]:
log_x = np.logspace(0, 4, 100)
log_y = log_x ** 0.5

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)

axes[0].plot(log_x, log_y, color="tab:blue")
axes[0].set_title("Linear axes")
axes[0].set_xlabel("x")
axes[0].set_ylabel("sqrt(x)")

axes[1].plot(log_x, log_y, color="tab:red")
axes[1].set_xscale("log")
axes[1].set_yscale("log")
axes[1].set_title("Log-log axes")
axes[1].set_xlabel("x (log)")
axes[1].set_ylabel("sqrt(x) (log)")

plt.show()


## 18. Styles, Themes, and `rcParams`

Consistent style matters when plots appear together in reports, notebooks, slide decks, or dashboards.

Matplotlib gives you three levels of control:
- built-in styles for quick changes
- temporary style contexts for one figure or section
- `rcParams` for global defaults across a notebook or script

The goal is consistency, not decoration for its own sake.


In [ ]:
print("Available sample of built-in styles:", plt.style.available[:8])

with plt.style.context("seaborn-v0_8-whitegrid"):
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(x, y, linewidth=2.3, color="tab:purple")
    ax.set_title("Temporary style context")
    ax.set_xlabel("x")
    ax.set_ylabel("sin(x)")
    plt.show()

plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11
print("Updated rcParams for titles and labels.")


## 19. Saving Figures and Exporting Properly

Notebook display and exported files are not always the same thing. A figure that looks acceptable inline can still export poorly if size, DPI, or bounding box settings are off.

Practical guidance:
- use PNG for raster output and PDF or SVG for vector output
- prefer vector formats for line art, reports, and publication workflows
- set DPI deliberately for raster exports
- use `bbox_inches="tight"` when labels or legends risk being clipped


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sales_df["date"], sales_df["sales"], color="tab:blue", linewidth=2)
ax.set_title("Figure saved to both PNG and SVG")
ax.set_ylabel("Sales")
ax.tick_params(axis="x", rotation=35)

png_path = export_dir / "sales_trend.png"
svg_path = export_dir / "sales_trend.svg"

fig.savefig(png_path, dpi=200, bbox_inches="tight", facecolor="white")
fig.savefig(svg_path, bbox_inches="tight", facecolor="white")
plt.show()

print("Saved:", png_path)
print("Saved:", svg_path)


## 20. Common Pitfalls and Best Practices

Common plotting problems are usually communication problems, not syntax problems.

Watch for:
- forgetting the Figure-Axes model and getting lost in multi-plot code
- overlapping labels or legends
- too many categories in one figure
- misleading axis limits
- overuse of color and decoration
- poor aspect ratios
- chart type mismatch
- missing units or unclear titles
- trusting defaults for a final figure

If you remember only a short review checklist before sharing a plot, use this one:
- can a new reader tell what is being compared?
- are the units and categories explicit?
- is any design choice exaggerating or hiding an effect?
- is the chart type helping the question rather than merely displaying data?


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)

# crowded / weaker version
axes[0].bar(
    [f"Category {i}" for i in range(1, 11)],
    rng.integers(20, 100, 10),
    color=["red", "blue", "green", "purple", "orange", "pink", "gray", "brown", "cyan", "olive"],
)
axes[0].set_title("Crowded and hard to scan")
axes[0].tick_params(axis="x", rotation=70)

# cleaner version
top_categories = pd.DataFrame(
    {"category": ["North", "West", "East", "South"], "value": [91, 84, 78, 70]}
)
axes[1].bar(top_categories["category"], top_categories["value"], color="steelblue")
axes[1].set_title("Focused, ordered, and readable")
axes[1].set_ylabel("Value")
axes[1].grid(axis="y", alpha=0.2)

plt.show()


## 21. End-to-End Visualization Workflow

This is where the notebook becomes more like real work.

A good plotting workflow often looks like this:
1. inspect the raw data
2. clean and derive the fields you need
3. create quick exploratory plots
4. choose the best final chart for the question
5. annotate the most important insight
6. polish labels, ticks, layout, and export settings

The example below starts with semi-processed revenue data and turns it into a small presentation-ready figure set.


In [ ]:
workflow_df = pd.DataFrame(
    {
        "month": pd.date_range("2025-01-01", periods=8, freq="MS"),
        "region": ["North", "South", "East", "West", "North", "South", "East", "West"],
        "revenue": ["120", "95", "110", "130", "140", "105", None, "150"],
        "orders": [42, 35, 39, 45, 48, 37, 40, 50],
    }
)

print("Initial profile:")
print(workflow_df.dtypes)
print(workflow_df.isna().sum())

workflow_df["revenue"] = pd.to_numeric(workflow_df["revenue"], errors="coerce")
workflow_df["revenue"] = workflow_df["revenue"].fillna(workflow_df["revenue"].median())
workflow_df["revenue_per_order"] = workflow_df["revenue"] / workflow_df["orders"]

monthly_total = workflow_df.groupby("month", as_index=False).agg(
    total_revenue=("revenue", "sum"),
    total_orders=("orders", "sum"),
)
monthly_total["rolling_2"] = monthly_total["total_revenue"].rolling(2).mean()

region_summary = workflow_df.groupby("region", as_index=False).agg(
    total_revenue=("revenue", "sum"),
    avg_revenue_per_order=("revenue_per_order", "mean"),
).sort_values("total_revenue", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), constrained_layout=True)

axes[0].plot(monthly_total["month"], monthly_total["total_revenue"], marker="o", linewidth=2.2, label="monthly revenue")
axes[0].plot(monthly_total["month"], monthly_total["rolling_2"], linewidth=2.2, linestyle="--", label="2-period rolling mean")
axes[0].set_title("Monthly revenue trend")
axes[0].set_ylabel("Revenue")
axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%b"))
axes[0].legend(frameon=False)
axes[0].grid(axis="y", alpha=0.2)

bars = axes[1].bar(region_summary["region"], region_summary["total_revenue"], color="darkcyan")
axes[1].set_title("Regional revenue comparison")
axes[1].set_ylabel("Total revenue")
axes[1].grid(axis="y", alpha=0.2)

best_bar = max(bars, key=lambda b: b.get_height())
axes[1].annotate(
    "Highest total revenue",
    (best_bar.get_x() + best_bar.get_width() / 2, best_bar.get_height()),
    xytext=(0, 10),
    textcoords="offset points",
    ha="center",
    arrowprops={"arrowstyle": "-|>", "color": "black"},
)

workflow_export = export_dir / "workflow_summary.pdf"
fig.savefig(workflow_export, bbox_inches="tight", facecolor="white")
plt.show()

print("Saved:", workflow_export)
print("\nregion_summary:")
print(region_summary)


## 22. Quick Reference Cheat Sheet

A short cheat sheet is useful because Matplotlib is broad enough that even experienced users do not memorize every method.


In [ ]:
cheat_sheet = pd.DataFrame(
    [
        ["Create figure + axes", "fig, ax = plt.subplots()", "Preferred starting point for most work"],
        ["Line plot", "ax.plot(x, y)", "Ordered trends and continuous sequences"],
        ["Scatter plot", "ax.scatter(x, y)", "Relationships between continuous variables"],
        ["Bar chart", "ax.bar(categories, values)", "Comparing discrete groups"],
        ["Histogram", "ax.hist(values, bins=...)", "Understanding distributions"],
        ["Annotation", "ax.annotate(...)", "Highlight important points or events"],
        ["Reference line", "ax.axhline(...) / ax.axvline(...)", "Thresholds, baselines, targets"],
        ["Layout", "plt.tight_layout()", "Quick spacing cleanup"],
        ["Export", "fig.savefig(path, dpi=..., bbox_inches='tight')", "Save presentation-ready output"],
        ["Style context", "with plt.style.context(...):", "Temporary style changes"],
    ],
    columns=["Task", "Typical call", "Use this when..."],
)

cheat_sheet
